<a href="https://colab.research.google.com/github/appleflavorMilk/machineRunnigstudy/blob/main/AngsanbleTrainML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

앙상블 학습:


*   정형데이터를 다루는데 가장특화된 알고리즘
*   대부분 결정트리를 기반으로 만들어짐



In [ ]:
#앙상블 학습의 대표:랜덤포레스트
#결정트리를 랜덤하게 만들고 각 결정트리의 예측을 이용
#결정트리를 만딜기 위해 데이터를 랜덤하게 만드는 방법 부트스트랩: 데이터 세트에서 중복을 허용해서 데이터를 샘플링하는 방식
#각 노드를 분할 할 때 기준을 전체특성중 일부분의 특성을 무작위로 선택함. 이때 선택하는 특성의 수는 보통 전체 특성의 제곱근->RandomForestClassifier
#단,회귀 모델인 RandomForestRegressor는 전체 특성 선택


사이킷런의 랜덤포레스트는 기본적으로 100개의 트리를 위와 같은 방법으로 만듦

*   분류예제인 경우: 각 트리별 클래스의 확률을 평균하여 가장높은 확률을 가진 클래스를 예측으로 삼는다.
*   회귀예제인 경우: 단순히 각 트리의 예측을 평균한다.




In [2]:
#데이터 받아오고 학습 훈련데이터로 분류
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
wine = pd.read_csv('https://bit.ly/wine_csv_data')
data = wine[['alcohol','sugar','pH']]
target = wine['class']
train_input,test_input,train_target,test_target = train_test_split(data,target,test_size = 0.2,random_state=42)

In [7]:
from sklearn.model_selection import cross_validate#교차 검증 수행
from sklearn.ensemble import RandomForestClassifier
import numpy as np # numpy import 추가
rf = RandomForestClassifier(n_jobs=-1,random_state = 42)
scores = cross_validate(rf,train_input,train_target,return_train_score=True,n_jobs=-1)#RandomForestClassifer은 보통 100개의 트리를 만드므로 모든 cPU사용할것
print(np.mean(scores['train_score']),np.mean(scores['test_score']))

0.9973541965122431 0.8905151032797809


In [8]:
#랜덤포레스트는 결정트리의 앙상블이기 때문에 DecisionTreeClassifier가 제공하는 중요 매개변수 모두제공
#criterion,max_depth,max_features,min_samples_slpit,min_impurity_decrease,min_sample_leaf 등
#랜덤 포레스트의 가장큰장점은 특성 중요도를 계산한다는 것이다(feature_importances_)
rf.fit(train_input,train_target)
print(rf.feature_importances_)

[0.23167441 0.50039841 0.26792718]


In [9]:
#RandomForestClassifier은 부트스트랩에 포함되지않고 남은 데이터(OOB)를 가지고 스스로 모델을 평가할수있다
rf = RandomForestClassifier(oob_score = True,n_jobs=-1,random_state=42)
rf.fit(train_input,train_target)
print(rf.oob_score_)#각 트리별 oob점수를 평균하여 출력

0.8934000384837406


엑스트라 트리: 랜덤포레스트와 비슷하지만 부트스트랩을 사용하진 않는다. 대신 바로 전체 훈련세트를 사용. 대신 노드를 분할할때 무작위로 분할

In [12]:
from sklearn.ensemble import ExtraTreesClassifier
et = ExtraTreesClassifier(n_jobs=-1,random_state = 42)
scores = cross_validate(et,train_input,train_target,return_train_score = True,n_jobs=-1)
print(np.mean(scores['train_score']),np.mean(scores['test_score']))

0.9974503966084433 0.8887848893166506


In [14]:
#특성 중요도 제공함
et.fit(train_input,train_target)
print(et.feature_importances_)

[0.20183568 0.52242907 0.27573525]


GradientBoostingClassifier:


*   깊이가 얕은 결정트리를 사용하여 이진트리의 오차를 줄이는 식
*   기본적으로 깊이가 3인 100개의 결정트리를 사용, 과대적합에 강하고 높은 일반화 성능



In [15]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state = 42)
scores = cross_validate(gb,train_input,train_target,return_train_score = True,n_jobs=-1)
print(np.mean(scores['train_score']),np.mean(scores['test_score']))

0.8881086892152563 0.8720430147331015


In [16]:
gb = GradientBoostingClassifier(n_estimators = 500,learning_rate = 0.2, random_state = 42)#결정트리 개수를 5배로 늘리고 학습률을 0.1(기본)에서 0.2로 늘림
scores = cross_validate(gb,train_input,train_target,return_train_score = True,n_jobs=-1)
print(np.mean(scores['train_score']),np.mean(scores['test_score']))

0.9464595437171814 0.8780082549788999


In [17]:
gb.fit(train_input,train_target)
print(gb.feature_importances_)
#GradientBoostingClassifier역시 특성 중요도를 나타냄

[0.15887763 0.6799705  0.16115187]


히스토그램 기반 그래디언트 부스팅


*   정형 데이터 기반 알고리즘중 가장 인기가 많음
*   입력 특성을 256개의 구간으로 나눕니다.
*   HistGradientBoostingClassifier은 트리의 개수를 n_estimators대신에 max_iter을 사용한다.



In [21]:
from sklearn.ensemble import HistGradientBoostingClassifier
hgb = HistGradientBoostingClassifier(random_state = 42)
scores = cross_validate(hgb,train_input,train_target,return_train_score=True,n_jobs=-1)
print(np.mean(scores['train_score']),np.mean(scores['test_score']))

0.9321723946453317 0.8801241948619236


In [22]:
#히스토기반 그래디언트 부스팅은 특성 중요도를 제공하지 않습니다.
#이런 경우 permutaion_importance()함수로 특성중요도를 계산 가능, 특성을 랜덤하게 섞어서 모델의 성능이 변화하는지 관찰해서 특성 중요도를 계산함
#n_repeats매개변수는 랜덤하게 섞을 횟수 설정. 기본값은 5
from sklearn.inspection import permutation_importance
hgb.fit(train_input,train_target)
result = permutation_importance(hgb,train_input,train_target,n_repeats=10,random_state=42,n_jobs=-1)
print(result.importances_mean)

[0.08876275 0.23438522 0.08027708]


In [24]:
#sklearn말고도 히스토그램그래디언트부스팅 알고리즘을 구현한 라이브러리가 있다. XGBoost
from xgboost import XGBClassifier
xgb = XGBClassifier(tree_method = 'hist',random_state = 42)
scores = cross_validate(xgb,train_input,train_target,return_train_score = True,n_jobs=-1)
print(np.mean(scores['train_score']),np.mean(scores['test_score']))

0.9572351925731766 0.8781988968682904


마이크로소프트에서 만든lightgbm역시 히스토그램그래디언트 부스팅알고리즘을 제공한다.

In [25]:
from lightgbm import LGBMClassifier
lgb = LGBMClassifier(random_state=42)
scores = cross_validate(lgb,train_input,train_target,return_train_score = True,n_jobs=-1)
print(np.mean(scores['train_score']),np.mean(scores['test_score']))

0.935828414851749 0.8801251203079884
